# ⚡ QuantumEnergyOS — Demo Interactiva
**Desde Mexicali, Baja California — para el noroeste y el mundo.**

Este notebook integra los tres módulos Q# del proyecto con simulaciones Python y visualizaciones interactivas:

| Módulo | Descripción |
|---|---|
| 🔌 `BalancearRed` | Optimización QAOA de red eléctrica regional |
| 🌡️ `FusionSim` | Simulación cuántica de plasma D-T |
| 🔗 `BraidingDebug` | Depuración de braiding topológico de Majorana |

---
> **Requisitos**: `pip install qsharp qiskit qiskit-aer matplotlib numpy scipy`

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Qiskit (simulaciones Python puras como respaldo)
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector

# Estilo visual
plt.style.use('dark_background')
COLORS = ['#00FFB3', '#FF6B6B', '#4ECDC4', '#FFE66D', '#A8EDEA']
print('✅ Imports OK')

---
## 🔌 Módulo 1 — Balanceo de Red Eléctrica (QAOA)

Simulamos la optimización de distribución de carga entre **4 nodos regionales**:
Mexicali · Tijuana · Ensenada · Tecate

**Algoritmo**: QAOA (Quantum Approximate Optimization Algorithm) con p=2 capas.

In [ ]:
# ── Construcción del circuito QAOA para balanceo de red ──────

def construir_qaoa_red(n_nodos=4, gamma=0.5, beta=0.3, p=2):
    """Construye circuito QAOA para optimización de red eléctrica."""
    qc = QuantumCircuit(n_nodos, n_nodos)
    qc.name = 'QAOA_BalanceoRed'

    # Estado inicial: superposición uniforme
    qc.h(range(n_nodos))
    qc.barrier(label='|+⟩')

    for capa in range(p):
        # Oráculo de carga: penalizar nodos adyacentes activos
        for i in range(n_nodos - 1):
            qc.cx(i, i + 1)
            qc.rz(2 * gamma, i + 1)
            qc.cx(i, i + 1)
        qc.barrier(label=f'Oráculo p={capa+1}')

        # Mezclador QAOA
        for i in range(n_nodos):
            qc.rx(2 * beta, i)
        qc.barrier(label=f'Mezclador p={capa+1}')

    qc.measure(range(n_nodos), range(n_nodos))
    return qc

qc_red = construir_qaoa_red()
print(qc_red.draw(output='text', fold=80))

In [ ]:
# ── Simulación y visualización ────────────────────────────────

NODOS = ['Mexicali', 'Tijuana', 'Ensenada', 'Tecate']
SHOTS = 2048

sim = AerSimulator()
qc_t = transpile(qc_red, sim)
job = sim.run(qc_t, shots=SHOTS)
counts = job.result().get_counts()

# Top 8 configuraciones más probables
top = dict(sorted(counts.items(), key=lambda x: -x[1])[:8])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0D1117')

# ── Izquierda: histograma de estados ──
ax1 = axes[0]
ax1.set_facecolor('#161B22')
estados = list(top.keys())
probs = [v / SHOTS for v in top.values()]
bars = ax1.bar(estados, probs, color=COLORS[0], alpha=0.85, edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Configuración de nodos (0=inactivo, 1=activo)', color='white', fontsize=10)
ax1.set_ylabel('Probabilidad', color='white', fontsize=10)
ax1.set_title('⚡ QAOA — Distribución de estados de red', color=COLORS[0], fontsize=13, pad=12)
ax1.tick_params(colors='white', rotation=45)
for bar, prob in zip(bars, probs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{prob:.3f}', ha='center', va='bottom', color='white', fontsize=8)

# ── Derecha: mapa de red eléctrica ──
ax2 = axes[1]
ax2.set_facecolor('#161B22')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_aspect('equal')
ax2.axis('off')
ax2.set_title('🗺️ Red Eléctrica — Baja California', color=COLORS[0], fontsize=13, pad=12)

# Posiciones geográficas aproximadas
pos = {'Mexicali': (8, 8), 'Tijuana': (2, 7), 'Ensenada': (1, 3), 'Tecate': (5, 6)}
edges = [('Mexicali','Tijuana'), ('Tijuana','Tecate'), ('Tecate','Mexicali'), ('Tijuana','Ensenada')]

# Estado más probable
mejor_estado = max(counts, key=counts.get)
activos = {NODOS[i]: mejor_estado[-(i+1)] == '1' for i in range(4)}

for a, b in edges:
    x = [pos[a][0], pos[b][0]]
    y = [pos[a][1], pos[b][1]]
    color = COLORS[0] if (activos[a] and activos[b]) else '#444'
    ax2.plot(x, y, color=color, linewidth=2.5, alpha=0.7, zorder=1)

for nodo, (x, y) in pos.items():
    color = COLORS[0] if activos[nodo] else COLORS[1]
    circle = plt.Circle((x, y), 0.6, color=color, zorder=3, alpha=0.9)
    ax2.add_patch(circle)
    ax2.text(x, y - 1.1, nodo, ha='center', color='white', fontsize=9, fontweight='bold')
    estado_txt = '🟢 ON' if activos[nodo] else '🔴 OFF'
    ax2.text(x, y + 1.0, estado_txt, ha='center', color='white', fontsize=7)

ax2.text(5, 0.5, f'Config óptima: {mejor_estado} | P={counts[mejor_estado]/SHOTS:.3f}',
         ha='center', color=COLORS[3], fontsize=9)

plt.tight_layout()
plt.savefig('balanceo_red.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print(f'\n🏆 Configuración óptima: {mejor_estado} ({counts[mejor_estado]} de {SHOTS} shots)')

---
## 🌡️ Módulo 2 — Simulación de Fusión D-T

Simulamos la evolución temporal de un **plasma de Deuterio-Tritio** usando:
- Pares de Bell para modelar el entrelazamiento D-T
- Hamiltoniano de evolución bajo campo magnético
- Estimación de la **condición de Lawson cuántica** (energía > 0.6)

In [ ]:
# ── Construcción del circuito de fusión ──────────────────────

def construir_fusion_dt(n_pares=3, tiempo=np.pi/2, campo_B=1.0):
    """Simula plasma D-T: pares Bell + evolución temporal."""
    n = n_pares * 2
    qc = QuantumCircuit(n, n)
    qc.name = 'FusionDT'

    # Pares entrelazados D-T (Bell states)
    for i in range(0, n, 2):
        qc.h(i)
        qc.cx(i, i + 1)
    qc.barrier(label='Pares D-T')

    # Evolución temporal del plasma
    angulo = 2.0 * tiempo * campo_B
    for q in range(n):
        qc.ry(angulo, q)
        qc.rz(angulo / 2, q)
    for i in range(n - 1):
        qc.cx(i, i + 1)
        qc.rz(angulo / 4, i + 1)
        qc.cx(i, i + 1)
    qc.barrier(label='Evolución')

    qc.measure(range(n), range(n))
    return qc

# Barrido de tiempos para curva de energía
tiempos = np.linspace(0, 2 * np.pi, 40)
energias = []

for t in tiempos:
    qc_f = construir_fusion_dt(tiempo=t)
    qc_t = transpile(qc_f, sim)
    result = sim.run(qc_t, shots=512).result().get_counts()
    total = sum(result.values())
    ones = sum(v * bin(int(k, 2)).count('1') / 6 for k, v in result.items())
    energias.append(ones / total)

print(f'✅ Barrido temporal completo: {len(tiempos)} puntos simulados')
print(f'   Energía máxima: {max(energias):.4f}')
print(f'   Energía mínima: {min(energias):.4f}')

In [ ]:
# ── Visualización de fusión ───────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0D1117')

# ── Izquierda: curva de energía del plasma ──
ax1 = axes[0]
ax1.set_facecolor('#161B22')

ax1.plot(tiempos / np.pi, energias, color=COLORS[1], linewidth=2.5, label='Energía plasma')
ax1.axhline(y=0.6, color=COLORS[3], linestyle='--', linewidth=1.5, label='Umbral de ignición (0.6)')
ax1.fill_between(tiempos / np.pi, energias, 0.6,
                 where=[e > 0.6 for e in energias],
                 alpha=0.2, color=COLORS[0], label='Zona de ignición')
ax1.set_xlabel('Tiempo de pulso (× π)', color='white', fontsize=10)
ax1.set_ylabel('Energía media del plasma', color='white', fontsize=10)
ax1.set_title('🌡️ Fusión D-T — Evolución temporal del plasma', color=COLORS[1], fontsize=13, pad=12)
ax1.tick_params(colors='white')
ax1.legend(facecolor='#161B22', edgecolor='gray', labelcolor='white', fontsize=9)
ax1.set_ylim(0, 1)

# Marcar máximo
idx_max = np.argmax(energias)
ax1.annotate(f'Máx: {energias[idx_max]:.3f}',
             xy=(tiempos[idx_max]/np.pi, energias[idx_max]),
             xytext=(tiempos[idx_max]/np.pi + 0.2, energias[idx_max] + 0.05),
             arrowprops=dict(arrowstyle='->', color=COLORS[3]),
             color=COLORS[3], fontsize=9)

# ── Derecha: estado cuántico de un par D-T en el punto óptimo ──
ax2 = axes[1]
ax2.set_facecolor('#161B22')

t_optimo = tiempos[idx_max]
qc_sv = QuantumCircuit(2)
qc_sv.h(0)
qc_sv.cx(0, 1)
angulo_opt = 2.0 * t_optimo
qc_sv.ry(angulo_opt, 0)
qc_sv.ry(angulo_opt, 1)
qc_sv.rz(angulo_opt / 2, 0)
qc_sv.rz(angulo_opt / 2, 1)

sv = Statevector(qc_sv)
probs_sv = sv.probabilities_dict()

estados_sv = list(probs_sv.keys())
probs_vals = list(probs_sv.values())
labels_sv = ['|D↑T↑⟩', '|D↑T↓⟩', '|D↓T↑⟩', '|D↓T↓⟩']

bars2 = ax2.bar(labels_sv[:len(probs_vals)], probs_vals,
                color=[COLORS[i % len(COLORS)] for i in range(len(probs_vals))],
                alpha=0.85, edgecolor='white', linewidth=0.5)
ax2.set_ylabel('Probabilidad', color='white', fontsize=10)
ax2.set_title(f'⚛️  Par D-T en t={t_optimo/np.pi:.2f}π (punto óptimo)', color=COLORS[1], fontsize=13, pad=12)
ax2.tick_params(colors='white')
ax2.set_ylim(0, 1)
for bar, prob in zip(bars2, probs_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{prob:.3f}', ha='center', va='bottom', color='white', fontsize=9)

plt.tight_layout()
plt.savefig('fusion_sim.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()

ignicion = energias[idx_max] > 0.6
print(f'\n{"✅ Ignición cuántica alcanzada" if ignicion else "⚠️  Energía insuficiente"}')
print(f'   Energía máxima: {energias[idx_max]:.4f} en t={t_optimo/np.pi:.3f}π')

---
## 🔗 Módulo 3 — Braiding Topológico de Majorana

Simulamos el intercambio (braid) de **anyones de Majorana** — la base de la computación cuántica topológica fault-tolerant.

**Por qué importa para energía**: los qubits de Majorana son naturalmente resistentes al ruido, lo que haría posible controladores cuánticos robustos para redes eléctricas y reactores de fusión.

In [ ]:
# ── Circuito de braiding topológico ──────────────────────────

def braid_majorana(qc, a, b):
    """Operación de intercambio σᵢ entre anyones a y b."""
    qc.s(a)
    qc.h(b)
    qc.cx(a, b)
    qc.h(b)
    qc.sdg(a)

def circuito_braiding(n_anyones=4, n_braids=3):
    """Secuencia de braids sobre N anyones con verificación de invarianza."""
    qc = QuantumCircuit(n_anyones, n_anyones)
    qc.name = 'BraidingMajorana'

    # Estado inicial: superposición
    qc.h(range(n_anyones))
    qc.barrier(label='Init')

    # Secuencia de braids: derecha → izquierda
    for _ in range(n_braids):
        for i in range(n_anyones - 1):
            braid_majorana(qc, i, i + 1)
        qc.barrier(label=f'Braid →')

    # Inverso: verificar invarianza topológica
    for _ in range(n_braids):
        for i in reversed(range(n_anyones - 1)):
            # Adjunto del braid
            qc.s(i)
            qc.h(i + 1)
            qc.cx(i, i + 1)
            qc.h(i + 1)
            qc.sdg(i)
        qc.barrier(label=f'Braid ←')

    # Deshacer superposición inicial
    qc.h(range(n_anyones))
    qc.measure(range(n_anyones), range(n_anyones))
    return qc

qc_braid = circuito_braiding()
print(qc_braid.draw(output='text', fold=100))

In [ ]:
# ── Simulación y análisis de invarianza topológica ───────────

# Correr múltiples barridos de profundidad de braid
profundidades = range(1, 8)
fidelidades = []

for p in profundidades:
    qc_p = circuito_braiding(n_braids=p)
    qc_t = transpile(qc_p, sim)
    counts_p = sim.run(qc_t, shots=1024).result().get_counts()
    # Fidelidad = fracción de shots que regresan a |0000⟩
    fidelidad = counts_p.get('0000', 0) / 1024
    fidelidades.append(fidelidad)

# Correr circuito principal
qc_bt = transpile(qc_braid, sim)
counts_b = sim.run(qc_bt, shots=2048).result().get_counts()
top_b = dict(sorted(counts_b.items(), key=lambda x: -x[1])[:6])

# ── Visualización ────────────────────────────────────────────
fig = plt.figure(figsize=(17, 6))
fig.patch.set_facecolor('#0D1117')
gs = GridSpec(1, 3, figure=fig, wspace=0.35)

# Panel 1: Fidelidad vs profundidad de braid
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor('#161B22')
ax1.plot(list(profundidades), fidelidades, 'o-',
         color=COLORS[2], linewidth=2.5, markersize=8, markerfacecolor=COLORS[3])
ax1.axhline(y=0.95, color=COLORS[0], linestyle='--', linewidth=1.5, label='Umbral topológico (0.95)')
ax1.set_xlabel('Profundidad de braid (p)', color='white', fontsize=10)
ax1.set_ylabel('Fidelidad |0000⟩', color='white', fontsize=10)
ax1.set_title('🔗 Invarianza Topológica\nvs Profundidad de Braid', color=COLORS[2], fontsize=11, pad=10)
ax1.tick_params(colors='white')
ax1.legend(facecolor='#161B22', edgecolor='gray', labelcolor='white', fontsize=8)
ax1.set_ylim(0, 1.05)
ax1.set_xticks(list(profundidades))

# Panel 2: Distribución de estados del braid
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor('#161B22')
estados_b = list(top_b.keys())
probs_b = [v / 2048 for v in top_b.values()]
bar_colors = [COLORS[0] if e == '0000' else COLORS[1] for e in estados_b]
bars3 = ax2.bar(estados_b, probs_b, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Estado medido', color='white', fontsize=10)
ax2.set_ylabel('Probabilidad', color='white', fontsize=10)
ax2.set_title('📊 Distribución Post-Braid\n(Top 6 estados)', color=COLORS[2], fontsize=11, pad=10)
ax2.tick_params(colors='white', rotation=45)
for bar, prob in zip(bars3, probs_b):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{prob:.3f}', ha='center', va='bottom', color='white', fontsize=8)
verde = mpatches.Patch(color=COLORS[0], label='|0000⟩ — invariante')
rojo  = mpatches.Patch(color=COLORS[1], label='Otros — error')
ax2.legend(handles=[verde, rojo], facecolor='#161B22', edgecolor='gray', labelcolor='white', fontsize=8)

# Panel 3: Diagrama de trenzas (braid diagram)
ax3 = fig.add_subplot(gs[2])
ax3.set_facecolor('#161B22')
ax3.set_xlim(0, 10)
ax3.set_ylim(0, 10)
ax3.axis('off')
ax3.set_title('🧵 Diagrama de Trenzas\nAnyones de Majorana', color=COLORS[2], fontsize=11, pad=10)

n_a = 4
xs = [2, 4, 6, 8]
y_start, y_end = 9.0, 1.0
braid_steps = [(0,1), (2,3), (1,2), (0,1), (2,3)]

trayectorias = {i: [(xs[i], y_start)] for i in range(n_a)}
y_curr = y_start
paso_h = (y_start - y_end) / (len(braid_steps) + 1)

for (a, b) in braid_steps:
    y_curr -= paso_h
    # Intercambio: a va a posición de b, b va a posición de a
    xa_old = trayectorias[a][-1][0]
    xb_old = trayectorias[b][-1][0]
    trayectorias[a].append((xb_old, y_curr))
    trayectorias[b].append((xa_old, y_curr))
    # Los demás continúan
    for i in range(n_a):
        if i != a and i != b:
            trayectorias[i].append((trayectorias[i][-1][0], y_curr))

for i in range(n_a):
    trayectorias[i].append((trayectorias[i][-1][0], y_end))

anyon_colors = [COLORS[0], COLORS[1], COLORS[2], COLORS[3]]
for i in range(n_a):
    pts = trayectorias[i]
    xs_t = [p[0] for p in pts]
    ys_t = [p[1] for p in pts]
    ax3.plot(xs_t, ys_t, color=anyon_colors[i], linewidth=3, alpha=0.9,
             label=f'γ{i+1}')
    ax3.plot(xs_t[0], ys_t[0], 'o', color=anyon_colors[i], markersize=10)
    ax3.plot(xs_t[-1], ys_t[-1], 's', color=anyon_colors[i], markersize=10)
    ax3.text(xs_t[0], ys_t[0] + 0.4, f'γ{i+1}', ha='center', color=anyon_colors[i], fontsize=9, fontweight='bold')

ax3.text(5, 0.2, '◦ = inicio   ■ = fin', ha='center', color='gray', fontsize=8)

plt.savefig('braiding_debug.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()

fid_principal = counts_b.get('0000', 0) / 2048
print(f'\n🔗 Fidelidad topológica: {fid_principal:.4f}')
print(f'{"✅ Invarianza topológica confirmada" if fid_principal > 0.8 else "⚠️  Ruido detectado — revisar profundidad"}')

---
## 📊 Resumen — Dashboard QuantumEnergyOS

Vista consolidada de los tres módulos en un solo panel.

In [ ]:
# ── Dashboard consolidado ─────────────────────────────────────

fig = plt.figure(figsize=(18, 5))
fig.patch.set_facecolor('#0D1117')
fig.suptitle('⚡ QuantumEnergyOS — Estado del Sistema',
             color='white', fontsize=16, fontweight='bold', y=1.02)

gs = GridSpec(1, 3, figure=fig, wspace=0.4)

metricas = [
    {
        'titulo': '🔌 Balanceo de Red',
        'subtitulo': 'Baja California',
        'valor': f'{counts[mejor_estado]/SHOTS:.1%}',
        'label': 'Probabilidad config. óptima',
        'config': mejor_estado,
        'color': COLORS[0],
        'nodos': NODOS,
        'activos': activos
    },
    {
        'titulo': '🌡️ Fusión D-T',
        'subtitulo': 'Plasma cuántico',
        'valor': f'{energias[idx_max]:.3f}',
        'label': 'Energía máxima del plasma',
        'tiempo': f't={t_optimo/np.pi:.2f}π',
        'color': COLORS[1],
        'ignicion': energias[idx_max] > 0.6
    },
    {
        'titulo': '🔗 Braiding Topológico',
        'subtitulo': 'Anyones Majorana',
        'valor': f'{fid_principal:.1%}',
        'label': 'Fidelidad topológica',
        'color': COLORS[2],
        'ok': fid_principal > 0.8
    }
]

for idx, m in enumerate(metricas):
    ax = fig.add_subplot(gs[idx])
    ax.set_facecolor('#161B22')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')

    # Título del módulo
    ax.text(5, 9.2, m['titulo'], ha='center', color=m['color'],
            fontsize=14, fontweight='bold')
    ax.text(5, 8.3, m['subtitulo'], ha='center', color='gray', fontsize=10)

    # Valor principal
    ax.text(5, 6.0, m['valor'], ha='center', color=m['color'],
            fontsize=36, fontweight='bold')
    ax.text(5, 4.8, m['label'], ha='center', color='white', fontsize=9)

    # Info adicional
    if idx == 0:
        nodos_on = [n for n, a in m['activos'].items() if a]
        ax.text(5, 3.5, f"Config: {m['config']}", ha='center', color='white', fontsize=10)
        ax.text(5, 2.6, f"Activos: {', '.join(nodos_on) if nodos_on else 'Ninguno'}",
                ha='center', color=m['color'], fontsize=9)
    elif idx == 1:
        ax.text(5, 3.5, m['tiempo'], ha='center', color='white', fontsize=10)
        estado_txt = '✅ Ignición' if m['ignicion'] else '⚠️ Sub-ignición'
        ax.text(5, 2.6, estado_txt, ha='center',
                color=COLORS[0] if m['ignicion'] else COLORS[3], fontsize=11)
    elif idx == 2:
        ax.text(5, 3.5, f'Mejor fidelidad: p=1', ha='center', color='white', fontsize=10)
        estado_txt = '✅ Invarianza OK' if m['ok'] else '❌ Revisar'
        ax.text(5, 2.6, estado_txt, ha='center',
                color=COLORS[0] if m['ok'] else COLORS[1], fontsize=11)

    # Barra de progreso
    valor_float = float(m['valor'].strip('%')) / 100 if '%' in m['valor'] else float(m['valor'])
    valor_norm = min(valor_float, 1.0)
    ax.barh(1.2, 8 * valor_norm, left=1, height=0.6,
            color=m['color'], alpha=0.7)
    ax.barh(1.2, 8, left=1, height=0.6,
            color='gray', alpha=0.15)
    # Borde del panel
    for spine_pos in [(0.1, 0.1, 9.8, 0.1), (0.1, 9.8, 9.8, 9.8),
                      (0.1, 0.1, 0.1, 9.8), (9.8, 0.1, 9.8, 9.8)]:
        ax.plot([spine_pos[0], spine_pos[2]], [spine_pos[1], spine_pos[3]],
                color=m['color'], alpha=0.3, linewidth=1)

plt.tight_layout()
plt.savefig('dashboard_quantum.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()

print('\n' + '='*60)
print('  ⚡ QuantumEnergyOS — Reporte de Estado')
print('='*60)
print(f'  🔌 Red eléctrica: config {mejor_estado} ({counts[mejor_estado]/SHOTS:.1%})')
print(f'  🌡️  Fusión D-T: energía máx = {energias[idx_max]:.4f}')
print(f'  🔗 Braiding: fidelidad = {fid_principal:.1%}')
print('='*60)
print('  📍 Desde Mexicali, BC — Kardashev 0 → 1')
print('='*60)